In [7]:
import jax
from jax import numpy as jnp
import pennylane as qml

from typing import Callable, List, Tuple, Union

import copy

In [8]:
import sys
sys.path.append('../')

from pqcqec.noise.simple_noise import PennylaneNoisyGates
from pqcqec.circuits.modify import  pennylane_state_embedding
from pqcqec.circuits.generate import generate_random_circuit

In [9]:
import jax.numpy as jnp

# ---------------------------
# 1) SU(2) from unit quaternion
# ---------------------------
def normalize_quaternion(q:jnp.ndarray, enforce_w_nonneg=True, eps=1e-12):
    q = jnp.asarray(q, dtype=jnp.float32)
    q = q / (jnp.linalg.norm(q) + eps)
    # Use JAX-friendly control flow: avoid Python boolean on traced values
    if enforce_w_nonneg:
        sign = jnp.where(q[0] < 0.0, -1.0, 1.0)
        q = q * sign
    return q  # (w,x,y,z), ||q||=1

def su2_from_quaternion(q):
    """U = w I - i (x σx + y σy + z σz) ∈ SU(2)."""
    w, x, y, z = normalize_quaternion(q)
    return jnp.array([[w - 1j * z, -1j * x - y],
                      [-1j * x + y, w + 1j * z]], dtype=jnp.complex64)

# ---------------------------
# 2) Canonical ZXZ decomposition (Rz-Rx-Rz) for compilation
#    Unique off the measure-zero boundaries β∈{0,π}
# ---------------------------
def _wrap_pi(t): return (t + jnp.pi) % (2 * jnp.pi) - jnp.pi

def zxz_from_su2(U, eps=1e-12):
    # strip global phase so det≈1
    U = U / jnp.sqrt(jnp.linalg.det(U))
    u11, u12, u21, u22 = U[0, 0], U[0, 1], U[1, 0], U[1, 1]

    beta = 2 * jnp.arccos(jnp.clip(jnp.abs(u11), 0.0, 1.0))  # β∈[0,π]

    cond_small = beta < eps
    cond_pi = jnp.pi - beta < eps

    alpha_small = _wrap_pi(jnp.angle(u11) - jnp.angle(u22))
    gamma_small = 0.0

    gamma_pi = _wrap_pi(jnp.angle(-u21) - jnp.angle(u12))

    apg = jnp.angle(u22) - jnp.angle(u11)    # α+γ
    amg = jnp.angle(-u21) - jnp.angle(u12)   # α-γ
    alpha_gen = _wrap_pi(0.5 * (apg + amg))
    gamma_gen = _wrap_pi(0.5 * (apg - amg))

    alpha = jnp.where(cond_small, alpha_small, jnp.where(cond_pi, 0.0, alpha_gen))
    beta_out = jnp.where(cond_small, 0.0, jnp.where(cond_pi, jnp.pi, beta))
    gamma = jnp.where(cond_small, gamma_small, jnp.where(cond_pi, gamma_pi, gamma_gen))
    return alpha, beta_out, gamma

# ---------------------------
# 3) End-to-end helper
# ---------------------------
def quaternion_to_zxz_angles(q):
    U = su2_from_quaternion(q)
    a, b, g = zxz_from_su2(U)
    return jnp.stack([a, b, g])



In [ ]:
class StateInputModelInterleavedQuaternionModel:
    """A class to define the Quaternion PQC model."""
    
    def __init__(self, circuit_ops:List, num_qubits:int, noise_model:PennylaneNoisyGates,
                 pqc_blocks=1, gate_blocks=1, seed=0):
        """
        Initialize the PQC model with the given parameters.
        Args:
            circuit_ops (List): List of circuit operations to be applied (circuit and its inverse).
            num_qubits (int): Number of qubits in the circuit.
            noise_model (PennylaneNoisyGates): Noise model to be applied.
            pqc_blocks (int): Number of PQC blocks.
            gate_blocks (int): Number of gates per block.
            seed (int): Random seed for parameter initialization.
        """

        self.num_qubits = num_qubits
        # self.pqc_arch = pennylane_PQC_RZRXRZ_unique
        self.circuit_ops = copy.deepcopy(circuit_ops)   
        self.num_angles = 3
        self.pqc_blocks = pqc_blocks
        self.gate_blocks = gate_blocks
        self.seed = seed
        self.noise_model = noise_model
        # self.uncomp_circuit = circuit_ops + circuit_ops[::-1]        # self.uncomp_circuit.extend([qml.adjoint(op) for op in self.circuit_ops[::-1]])
        self.num_gates = len(self.circuit_ops)

        self.qdev_cpu = qml.device("default.qubit", wires=self.num_qubits)
        self.diff_method = "backprop"  # Use backpropagation for differentiation

        self.pqc_gates = ['rz', 'rx', 'rz']
        self.num_pqc_angles = 3
        self.num_quanternion_values = 4

        self.param_sz = (int(self.pqc_blocks * jnp.ceil(self.num_gates/self.gate_blocks)), self.num_qubits, self.num_quanternion_values)

        # self.pqc_params = jax.random.uniform(jax.random.PRNGKey(self.seed), self.param_sz, jnp.float32, -jnp.pi, jnp.pi)
        self.quaternions = jnp.zeros( self.param_sz, dtype=jnp.float32).at[..., 0].set(1.0)
        # print(self.quaternions)
        # self.pqc_params = pnp.array(init_params, requires_grad=True, dtype=jnp.float32)


        
        @qml.qnode(self.qdev_cpu, interface='jax', diff_method=self.diff_method)
        def model_circuit(state, pqc_params):
            """Define the PQC model circuit."""
            # 1) Apply state embedding:
            pennylane_state_embedding(state, self.num_qubits)

            # @qml.for_loop(0, self.num_gates)
            for i, op in enumerate(self.circuit_ops):
            # def loop_body(i):
                gate, qubit, param = op
                # Apply the noisy gate:
                # if not param:
                self.noise_model.apply_gate(gate, qubit, angle=param)

                # Apply PQC to the qubit:
                if (i+1) % self.gate_blocks == 0:
                    # 2) Apply the PQC gates:
                    # print(f"Applying PQC block {i // self.gate_blocks + 1} with params: {pqc_params[i // self.gate_blocks]}")
                    pqc_params_block = pqc_params[i // self.gate_blocks]
                    # self.pqc_arch(self.num_qubits, pqc_params_block)
                    for qubit in range(self.num_qubits):
                        for j, pqc in enumerate(self.pqc_gates):
                            self.noise_model.apply_gate(pqc, qubit, angle=pqc_params_block[qubit, j])

            # 3) Return the output state:
            return qml.state()
        

        self.model_circuit = model_circuit
        self.batched_model_circuit = jax.jit(jax.vmap(self.model_circuit, in_axes=(0, None)))

    def get_pqc_params_from_block_quaternions(self, quaternions):
        """Convert quaternions to PQC parameters."""
        angles = jax.vmap(quaternion_to_zxz_angles)(quaternions)
        return angles

    def get_pqc_params_from_all_quaternions(self):
        """Convert all quaternions to PQC parameters."""
        block_angles = jax.vmap(self.get_pqc_params_from_block_quaternions)(self.quaternions)
        return block_angles
    
    def run_model_batch(self, in_state, params=None):
        """Run the model circuit on the BATCHED parameters and return the output state."""
        if params is None:
            params = self.get_pqc_params_from_block_quaternions(self.quaternions)
        return self.batched_model_circuit(in_state, params)

    def __call__(self, *args, **kwds):
        return self.run_model_batch(*args, **kwds)
    
    def __str__(self):
        return str(self.circuit_ops)

    def draw_mpl(self, in_state, params=None):
        """Draw the model circuit using matplotlib."""
 
        if params is None:
            params = self.get_pqc_params_from_block_quaternions(self.quaternions)

        print(f"Drawing circuit with params: {params}")
        print(f"Input state: {in_state}")
        print(f'Model: {self}')

        return qml.draw_mpl(self.model_circuit, decimals=4)(in_state, params)

    def get_circuit_tokens(self):
        """Get the circuit tokens."""
        tokens = []
        for i, op in enumerate(self.circuit_ops):
        # def loop_body(i):
            tokens.append(op)
            
            if (i+1) % self.gate_blocks == 0:
                # 2) Apply the PQC gates:
                # print(f"Applying PQC block {i // self.gate_blocks + 1} with params: {pqc_params[i // self.gate_blocks]}")
                pqc_params_block = self.get_pqc_params_from_block_quaternions(self.quaternions[i // self.gate_blocks])
                # Add PQC parameters to the tokens:
                for qubit in range(self.num_qubits):
                    for j, pqc in enumerate(self.pqc_gates):
                        tokens.append((pqc, [qubit], [pqc_params_block[qubit, j].item()]))

        # 3) Return the circuit tokens with PQC params:
        return tokens
    


In [11]:
circops = generate_random_circuit(num_qubits=3, num_gates=4, seed=0, backend='list')
print(circops)
model = StateInputModelInterleavedQuaternionModel(
    circuit_ops=circops,
    num_qubits=3,
    noise_model=PennylaneNoisyGates(),
    pqc_blocks=1,
    gate_blocks=1,
    seed=0
)

model.quaternions

[('cz', [2, 1], []), ('cx', [1, 2], []), ('h', [1], []), ('z', [1], [])]


Array([[[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]],

       [[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]],

       [[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]],

       [[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]]], dtype=float32)

In [12]:
model.get_pqc_params_from_all_quaternions()

Array([[[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]], dtype=float32)